# Along track distance

In [1]:
%load_ext autoreload
%autoreload 2

import os

import geopandas as gpd
import pandas as pd

import airbornegeo

os.environ["POLARTOOLKIT_EPSG"] = "3031"

/home/sungw937/airbornegeo/src/airbornegeo/levelling.py:21: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
data_df = pd.read_csv("../data/AGAP_gravity_survey.csv")
data_df = data_df[["easting", "northing", "line", "unixtime"]]
data_df = data_df.sort_values(["line", "unixtime"]).reset_index(drop=True)
data_df

,easting,northing,line,unixtime
0,1.000024e+06,226237.330771,1,1.229507e+09
1,1.000083e+06,226246.631269,1,1.229507e+09
2,1.000142e+06,226255.809132,1,1.229507e+09
3,1.000201e+06,226264.969079,1,1.229507e+09
4,1.000260e+06,226274.156809,1,1.229507e+09
...,...,...,...,...
332876,1.587964e+06,496507.950674,99,1.230382e+09
332877,1.587973e+06,496454.987752,99,1.230382e+09
332878,1.587983e+06,496401.908627,99,1.230382e+09
332879,1.587992e+06,496348.713306,99,1.230382e+09


In [3]:
# plot the survey colored by line number
airbornegeo.plotly_points(
    data_df[::10],
    color_col="line",
    robust=False,
    size=3,
    edge_width=None,
)

In [4]:
# turn to geopandas geodataframe
data_df = gpd.GeoDataFrame(
    data_df,
    geometry=gpd.points_from_xy(x=data_df.easting, y=data_df.northing),
    crs="EPSG:3031",
)

## Along track distance per line

In [5]:
data_df["line_distance"] = airbornegeo.along_track_distance(
    data_df,
    groupby_column="line",
)

airbornegeo.plotly_points(
    data_df[::10],
    color_col="line_distance",
    hover_cols=["line"],
    robust=False,
    size=3,
    edge_width=None,
)
data_df

,easting,northing,line,unixtime,geometry,line_distance
0,1.000024e+06,226237.330771,1,1.229507e+09,POINT (1000023.79 226237.331),0.000000
1,1.000083e+06,226246.631269,1,1.229507e+09,POINT (1000082.905 226246.631),59.842447
2,1.000142e+06,226255.809132,1,1.229507e+09,POINT (1000142.048 226255.809),119.693401
3,1.000201e+06,226264.969079,1,1.229507e+09,POINT (1000201.195 226264.969),179.545645
4,1.000260e+06,226274.156809,1,1.229507e+09,POINT (1000260.224 226274.157),239.285174
...,...,...,...,...,...,...
332876,1.587964e+06,496507.950674,99,1.230382e+09,POINT (1587964.113 496507.951),78906.532032
332877,1.587973e+06,496454.987752,99,1.230382e+09,POINT (1587973.377 496454.988),78960.299070
332878,1.587983e+06,496401.908627,99,1.230382e+09,POINT (1587982.56 496401.909),79014.166666
332879,1.587992e+06,496348.713306,99,1.230382e+09,POINT (1587991.661 496348.713),79068.134996


## Along track distance for the whole survey

In [6]:
data_df["survey_distance"] = airbornegeo.along_track_distance(
    data_df,
)

airbornegeo.plotly_points(
    data_df[::10],
    color_col="survey_distance",
    robust=False,
    size=3,
    edge_width=None,
)
data_df

,easting,northing,line,unixtime,geometry,line_distance,survey_distance
0,1.000024e+06,226237.330771,1,1.229507e+09,POINT (1000023.79 226237.331),0.000000,0.000000e+00
1,1.000083e+06,226246.631269,1,1.229507e+09,POINT (1000082.905 226246.631),59.842447,5.984245e+01
2,1.000142e+06,226255.809132,1,1.229507e+09,POINT (1000142.048 226255.809),119.693401,1.196934e+02
3,1.000201e+06,226264.969079,1,1.229507e+09,POINT (1000201.195 226264.969),179.545645,1.795456e+02
4,1.000260e+06,226274.156809,1,1.229507e+09,POINT (1000260.224 226274.157),239.285174,2.392852e+02
...,...,...,...,...,...,...,...
332876,1.587964e+06,496507.950674,99,1.230382e+09,POINT (1587964.113 496507.951),78906.532032,3.962065e+07
332877,1.587973e+06,496454.987752,99,1.230382e+09,POINT (1587973.377 496454.988),78960.299070,3.962070e+07
332878,1.587983e+06,496401.908627,99,1.230382e+09,POINT (1587982.56 496401.909),79014.166666,3.962076e+07
332879,1.587992e+06,496348.713306,99,1.230382e+09,POINT (1587991.661 496348.713),79068.134996,3.962081e+07


## Distances without time data
The above functions assume the data is sorted by time. If you don't have time data, but your data is still sorted by time, the functions will work. If you are unsure if your data is sorted by time, then you can use set `guess_start_position` to `True`. This will use which ever end of the segment is furthers left as the starting point and calculate distances relative to that point. 

In [7]:
# un sort the data to demonstrate
unsorted_data_df = data_df.sample(frac=1).reset_index(drop=True)
unsorted_data_df.head()

,easting,northing,line,unixtime,geometry,line_distance,survey_distance
0,1.282514e+06,502392.455542,90,1.229691e+09,POINT (1282513.887 502392.456),237866.206096,3.744190e+07
1,1.341897e+06,371645.610945,64,1.231065e+09,POINT (1341897.441 371645.611),33019.991811,2.757552e+07
2,1.269841e+06,313929.005142,18,1.229857e+09,POINT (1269841.357 313929.005),192337.426992,7.619132e+06
3,1.167241e+06,345584.588163,8,1.230576e+09,POINT (1167240.68 345584.588),104137.940206,2.870614e+06
4,1.147508e+06,361853.401768,4,1.230572e+09,POINT (1147508.367 361853.402),171256.930169,1.142715e+06


In [8]:
unsorted_data_df["line_distance"] = airbornegeo.along_track_distance(
    unsorted_data_df[::10],
    groupby_column="line",
    guess_start_position=True,
)

airbornegeo.plotly_points(
    unsorted_data_df,
    color_col="line_distance",
    robust=False,
    size=3,
    edge_width=None,
)
unsorted_data_df.head()

,easting,northing,line,unixtime,geometry,line_distance,survey_distance
0,1.282514e+06,502392.455542,90,1.229691e+09,POINT (1282513.887 502392.456),237773.262042,3.744190e+07
1,1.341897e+06,371645.610945,64,1.231065e+09,POINT (1341897.441 371645.611),NaN,2.757552e+07
2,1.269841e+06,313929.005142,18,1.229857e+09,POINT (1269841.357 313929.005),NaN,7.619132e+06
3,1.167241e+06,345584.588163,8,1.230576e+09,POINT (1167240.68 345584.588),NaN,2.870614e+06
4,1.147508e+06,361853.401768,4,1.230572e+09,POINT (1147508.367 361853.402),NaN,1.142715e+06
